# NLP Forward Example

Define NLP data, training, and projection settings in the notebook, then run the same builder-owned runner used by `main.py`.


## Setup
Import the builder runner and create helpers for notebook-local run folders and summaries.


In [1]:
from argparse import Namespace
from pathlib import Path
import json

from kkthn.builder import ProblemBuilder


ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def make_output_root(name):
    output_root = ROOT / "notebooks" / "_runs" / name
    output_root.mkdir(parents=True, exist_ok=True)
    return output_root


def latest_run_dir(output_root):
    runs = [path for path in output_root.iterdir() if path.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No run directories found under {output_root}")
    return max(runs, key=lambda path: path.stat().st_mtime)


def load_summary(run_dir):
    with open(run_dir / "summary.json", "r", encoding="utf-8") as fh:
        return json.load(fh)


def standard_args(problem_type, data, config, output_root):
    return Namespace(
        type=problem_type,
        action="run",
        mode="forward",
        p=data.get("p"),
        n=data.get("n"),
        me=data.get("me"),
        mi=data.get("mi"),
        samples=data.get("num_samples", data.get("N_points", data.get("N_samples"))),
        epochs=config.get("epochs"),
        batch_size=config.get("batch_size"),
        learning_rate=config.get("learning_rate"),
        solver=data.get("solver"),
        train_frac=config.get("train_frac"),
        hidden_size=config.get("hidden_size"),
        hidden_layers=config.get("hidden_layers"),
        seed=data.get("seed"),
        noise_scale=data.get("noise_scale"),
        output_dir=str(output_root),
    )


## Configuration
Edit these dictionaries to change the generated problem, training loop, or projection settings.


In [2]:
DATA = {
    "type": "nlp",
    "n_x": 2,
    "n_y": 4,
    "n_eq": 1,
    "n_ineq": 1,
    "N_samples": 12,
    "N_points": 12,
    "seed": 42,
    "noise_scale": 0.0,
    "solver": "SCS",
    "is_diag_Q": False,
    "q_diag_shift": 0.5,
    "nl_margin": 1.0,
    "bound_margin": 1.0,
    "bound_scale": 0.2,
    "param_scale": 0.4,
    "force_regenerate": True,
    "x_L": [-1.0],
    "x_U": [1.0],
}

CONFIG = {
    "epochs": 3,
    "batch_size": 4,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 32,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}

PROJ = {
    "fb_eps": 1e-8,
    "gn_max_iters": 25,
    "gn_tol": 1e-8,
}


## Train
Create CLI-like arguments and call `ProblemBuilder.run(...)` with the notebook dictionaries.


In [3]:
output_root = make_output_root("nlp_forward")
args = standard_args("nlp", DATA, CONFIG, output_root)
exit_code = ProblemBuilder.run(args, root=ROOT, data=DATA, train=CONFIG, proj=PROJ)
run_dir = latest_run_dir(output_root)
summary_json = load_summary(run_dir)

print(f"Exit code: {exit_code}")
print(f"Run directory: {run_dir}")
print(f"Saved artifacts: {summary_json['artifacts']}")


KKT-HardNet runner | type=nlp action=run mode=forward
Case directory: D:\Projects\KKTHardNet\case\nlp
Data config: {'type': 'nlp', 'n_x': 2, 'n_y': 4, 'n_eq': 1, 'n_ineq': 1, 'N_samples': 12, 'N_points': 12, 'seed': 42, 'noise_scale': 0.0, 'solver': 'SCS', 'is_diag_Q': False, 'q_diag_shift': 0.5, 'nl_margin': 1.0, 'bound_margin': 1.0, 'bound_scale': 0.2, 'param_scale': 0.4, 'force_regenerate': True, 'x_L': [-1.0, -1.0], 'x_U': [1.0, 1.0]}
Train config: {'epochs': 3, 'batch_size': 4, 'learning_rate': 0.001, 'train_frac': 0.8, 'hidden_size': 32, 'hidden_layers': 2, 'seed': 42, 'dtype': 'float64', 'print_every': 1, 'hidden_dim': 2}
Projection config: {'fb_eps': 1e-08, 'gn_max_iters': 25, 'gn_tol': 1e-08}
Labels: X=(12, 2) Y=(12, 4) source=optimizer
KKT-HardNet
  dims: n_x=2 n_y=4 n_eq=1 n_ineq=9
  mode: forward
  samples: train=9 val=3 batch_size=4
  network: [2, 32, 32, 4]
epoch 0001 | train=4.553543e-03 val=1.065372e-03 raw_val=5.256806e-01 eq=2.100e-10 ineq=1.016e-09 | train_batch=1.16

## Summary
Read the emitted `summary.json` from the newest run folder.


In [4]:
summary = {
    "output_dir": str(run_dir),
    "dims": summary_json["dims"],
    "final": summary_json["final"],
    "component_time_percent": summary_json["timing_profile"]["component_time_percent"],
}
print(json.dumps(summary, indent=2))


{
  "output_dir": "D:\\Projects\\KKTHardNet\\notebooks\\_runs\\nlp_forward\\nlp_20260414_121546",
  "dims": {
    "n_eq": 1,
    "n_ineq": 9,
    "n_x": 2,
    "n_y": 4,
    "n_z": 23,
    "raw_n_ineq": 1
  },
  "final": {
    "epoch": 3,
    "mean_batch_loss": 0.0014520526343649403,
    "train_batches": 3,
    "train_epoch_time_sec": 0.00674680000000194,
    "train_eq_l2": 7.261418010094821e-11,
    "train_eval_time_sec": 0.0026653999999979305,
    "train_ineq_l2": 1.1841251337917388e-09,
    "train_loss": 0.0009153774544090811,
    "train_raw_mse": 0.5591461279511283,
    "train_step_time_per_batch_sec": 0.0017382666666681719,
    "train_step_time_sec": 0.005214800000004516,
    "train_time_per_batch_sec": 0.00224893333333398,
    "val_eq_l2": 7.864666325592869e-11,
    "val_ineq_l2": 2.437401304847693e-09,
    "val_loss": 9.957927786616453e-05,
    "val_raw_mse": 0.5591344484038394,
    "validation_batches": 1,
    "validation_epoch_time_sec": 0.002593599999997309,
    "validation_t